# Exploratory Data Analysis — Dubai Office Hotspots

EDA on the 165-district engineered dataset (`FINAL_DATASET.csv`). Goal: understand feature
distributions, relationships, multicollinearity, and the hotspot label before modelling.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

DATA_PATH = "/content/drive/MyDrive/dataset/FINAL_DATASET.csv"   # Colab; or "../DATA/FINAL_DATASET.csv" locally
df = pd.read_csv(DATA_PATH)

features = ['avg_sale_price','rental_yield','transaction_count','contract_count',
           'avg_rent','mall_score','metro_score','parking_score']

# Optional: attach district names for the labelled charts (skip silently if not present)
try:
    _n = pd.read_csv("/content/drive/MyDrive/dataset/area_names.csv")
    _n['area_id'] = _n['area_id'].astype(float).astype(int)
    df['area_id'] = df['area_id'].astype(float).astype(int)
    df = df.merge(_n.drop_duplicates('area_id')[['area_id','area_name_en']], on='area_id', how='left')
except Exception:
    pass
df['label'] = df.get('area_name_en', df['area_id'].astype(str))

print("Shape:", df.shape)
df.head()


## 1. Summary statistics


In [ ]:
df[features].describe().T.round(3)


## 2. Data quality — missing values


In [ ]:
print("Missing values per column:")
print(df[features].isnull().sum())
print("\nDuplicate area_id rows:", df['area_id'].duplicated().sum())


## 3. Feature distributions

Several features are heavily right-skewed (a few high-value districts), which is why we min-max
scale before scoring/modelling.


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, f in zip(axes.ravel(), features):
    sns.histplot(df[f], kde=True, ax=ax, color="steelblue")
    ax.set_title(f)
plt.tight_layout(); plt.show()


## 4. Outliers (boxplots)


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 7))
for ax, f in zip(axes.ravel(), features):
    sns.boxplot(y=df[f], ax=ax, color="lightcoral")
    ax.set_title(f)
plt.tight_layout(); plt.show()


## 5. Correlation heatmap

**Watch for multicollinearity.** `metro_score` and `mall_score` are almost the same signal here —
districts with a nearby metro almost always have a nearby mall — so they carry largely redundant
information. Worth noting (and a candidate to merge into one 'accessibility' feature).


In [ ]:
corr = df[features].corr()
plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True)
plt.title("Feature correlation"); plt.tight_layout(); plt.show()

# print the strongest pairs
import itertools
pairs = [(a, b, corr.loc[a, b]) for a, b in itertools.combinations(features, 2)]
print("Strongest correlations:")
for a, b, v in sorted(pairs, key=lambda x: -abs(x[2]))[:5]:
    print(f"  {a:<18} ~ {b:<18} {v:+.3f}")


## 6. Hotspot score & label balance

The label = top 30% of `hotspot_score`. Classes are imbalanced (~30% positive), which is why the
models stratify the split and use class weights.


In [ ]:
df['hotspot_label'] = (df['hotspot_score'] >= df['hotspot_score'].quantile(0.70)).astype(int)

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
sns.histplot(df['hotspot_score'], bins=30, kde=True, ax=ax[0], color="seagreen")
ax[0].axvline(df['hotspot_score'].quantile(0.70), ls='--', color='red', label='70th pct (label cutoff)')
ax[0].set_title("Hotspot score distribution"); ax[0].legend()
df['hotspot_label'].value_counts().rename({0:'Not hotspot',1:'Hotspot'}).plot.bar(ax=ax[1], color=['#bdc3c7','#2980b9'])
ax[1].set_title("Label balance"); ax[1].set_ylabel("districts"); ax[1].tick_params(axis='x', rotation=0)
plt.tight_layout(); plt.show()
print(df['hotspot_label'].value_counts().to_dict())


## 7. Top & bottom districts


In [ ]:
top = df.nlargest(15, 'hotspot_score'); bot = df.nsmallest(15, 'hotspot_score')
fig, ax = plt.subplots(1, 2, figsize=(16, 6))
sns.barplot(data=top, y='label', x='hotspot_score', ax=ax[0], color="#2980b9")
ax[0].set_title("Top 15 districts"); ax[0].set_xlabel("hotspot score"); ax[0].set_ylabel("")
sns.barplot(data=bot, y='label', x='hotspot_score', ax=ax[1], color="#bdc3c7")
ax[1].set_title("Bottom 15 districts"); ax[1].set_xlabel("hotspot score"); ax[1].set_ylabel("")
plt.tight_layout(); plt.show()


## 8. Feature relationships


In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18, 5))
sns.scatterplot(data=df, x='avg_rent', y='rental_yield', hue='hotspot_label', palette='coolwarm', ax=ax[0])
ax[0].set_title("Rent vs rental yield")
sns.scatterplot(data=df, x='transaction_count', y='hotspot_score', hue='hotspot_label', palette='coolwarm', ax=ax[1])
ax[1].set_title("Market liquidity vs hotspot score")
sns.scatterplot(data=df, x='metro_score', y='mall_score', hue='hotspot_label', palette='coolwarm', ax=ax[2])
ax[2].set_title("Metro vs mall proximity (note the collinearity)")
plt.tight_layout(); plt.show()


## 9. Insights

- **Distributions** — `avg_rent`, `avg_sale_price`, `transaction_count` and `contract_count` are
  strongly right-skewed: a handful of prime districts dominate. Min-max scaling before scoring keeps
  these from swamping the composite.
- **Multicollinearity** — `metro_score` and `mall_score` correlate ~0.94 (near-duplicate). They could
  reasonably be merged into a single accessibility feature; keeping both slightly double-counts
  accessibility in the weighted score.
- **Label balance** — ~30% of districts are hotspots, so accuracy alone is misleading; F1/AUC and the
  stratified split + class weights in the models are the right choices.
- **Face validity** — the top districts (Business Bay, Dubai Marina, JLT, Downtown) are Dubai's real
  commercial cores, and the bottom districts are outlying residential areas — the score behaves
  sensibly.
- **Relationships** — hotspot districts cluster at higher liquidity and accessibility; rental yield is
  only weakly related to the score on its own, consistent with its modest survey weight.
